# Stage 1 - loading, de-duplication and the diversity gradient

Load each dataset's raw CAN frames into the canonical table, measure the duplication that inflates accuracy, strictly de-duplicate to the unique attack signatures, and split into train/test.

In [1]:
import sys
from pathlib import Path
try:
    import adversec
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent)); import adversec
import numpy as np, pandas as pd
from adversec import config
from adversec.contract import FEATURES, LABEL_COLUMN, ID_COLUMN, DATA_COLUMNS
pd.set_option('display.width', 140)

# The two datasets are treated identically
# The only dataset-specific code in the project is each dataset's loader (adversec/datasets/ciciov.py, road.py).
DATASETS = ['ciciov2024', 'road']

## Step 1 - load the raw canonical table
Each loader converts its raw format (CICIoV decimal CSVs, ROAD candump logs) into the same columns: `ID, DATA_0..DATA_7, true_class`.

In [2]:
from adversec.datasets.registry import get_dataset
raw = {}
for name in DATASETS:
    raw[name] = get_dataset(name).load()
    print(f'\n=== {name}: {len(raw[name]):,} frames ===')
    display(raw[name].head())


=== ciciov2024: 1,408,219 frames ===


,ID,DATA_0,DATA_1,DATA_2,DATA_3,DATA_4,DATA_5,DATA_6,DATA_7,true_class
0,65,96,0,0,0,0,0,0,0,benign
1,1068,132,13,160,0,0,0,0,0,benign
2,535,127,255,127,255,127,255,127,255,benign
3,131,15,224,0,0,0,0,0,0,benign
4,936,1,0,39,16,0,0,0,0,benign



=== road: 66,252 frames ===


,ID,DATA_0,DATA_1,DATA_2,DATA_3,DATA_4,DATA_5,DATA_6,DATA_7,true_class
0,208,2,115,4,100,137,255,110,0,max-speedometer
1,208,10,115,4,100,136,255,110,0,max-speedometer
2,208,18,115,4,100,135,255,110,0,max-speedometer
3,208,26,115,4,100,134,255,110,0,max-speedometer
4,208,34,115,4,100,132,255,111,0,max-speedometer


## Step 2 - audit duplication (the accuracy trap)
The fraction of rows that are redundant copies. High duplication is what inflates naive accuracy.

In [3]:
from adversec.pipeline import audit_duplication
dup_audit = {}
for name in DATASETS:
    a = audit_duplication(raw[name], FEATURES + [LABEL_COLUMN])
    dup_audit[name] = a
    print(f"{name:12s} rows={a['total_rows']:>10,}  unique={a['unique_signatures']:>8,}  duplication={a['duplication_rate_pct']:>7}%")

ciciov2024   rows= 1,408,219  unique=   3,588  duplication=99.7452%
road         rows=    66,252  unique=  39,858  duplication=39.8388%


## Step 3 - strict de-duplication -> the diversity gradient
One row per unique (features + class) signature. The per-class counts are the study's independent variable: how many genuinely distinct attacks each class holds.

In [4]:
from adversec.pipeline import strict_dedup
strict = {}
for name in DATASETS:
    strict[name] = strict_dedup(raw[name], FEATURES)
    print(f'\n=== {name}: {len(strict[name]):,} unique signatures ===')
    display(strict[name][LABEL_COLUMN].value_counts().rename('unique signatures').to_frame())


=== ciciov2024: 3,588 unique signatures ===


,unique signatures
true_class,
benign,3547
DoS,21
spoofing-RPM,10
spoofing-SPEED,5
spoofing-STEERING_WHEEL,3
spoofing-GAS,2



=== road: 39,858 unique signatures ===


,unique signatures
true_class,
benign,21188
max-speedometer,10559
reverse-light-on,5994
reverse-light-off,1525
fuzzing,592


## Step 4 - signature-level train/test split
Splitting happens on unique signatures BEFORE any augmentation, so no copy can straddle train and test. Tiny classes get a single-signature test floor.

In [5]:
from adversec.pipeline import split_train_test
split = {}
for name in DATASETS:
    tr, te = split_train_test(strict[name]); split[name] = (tr, te)
    print(f'\n=== {name}: train {len(tr):,} / test {len(te):,} ===')
    tbl = pd.DataFrame({'train': tr[LABEL_COLUMN].value_counts(), 'test': te[LABEL_COLUMN].value_counts()}).fillna(0).astype(int)
    display(tbl)


=== ciciov2024: train 2,870 / test 718 ===


,train,test
true_class,,
DoS,17,4
benign,2838,709
spoofing-GAS,1,1
spoofing-RPM,8,2
spoofing-SPEED,4,1
spoofing-STEERING_WHEEL,2,1



=== road: train 31,886 / test 7,972 ===


,train,test
true_class,,
benign,16950,4238
max-speedometer,8447,2112
reverse-light-on,4795,1199
reverse-light-off,1220,305
fuzzing,474,118


## Step 5 - save for the next stage
Written to `datasets/processed/<name>_*.csv` with a consistent per-dataset prefix (equal naming).

In [6]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    strict[name].to_csv(config.PROCESSED_DIR / f'{name}_strict.csv', index=False)
    split[name][0].to_csv(config.PROCESSED_DIR / f'{name}_train.csv', index=False)
    split[name][1].to_csv(config.PROCESSED_DIR / f'{name}_test.csv', index=False)
    print('saved', name)

saved ciciov2024
saved road


## Step 6 - save the dedup/split audit (for report writing)
Citable counts: raw duplication rate, per-class signature counts, and the train/test split sizes — one JSON per dataset, written to `results/<name>_dedup_audit.json`.

In [7]:
import json
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in DATASETS:
    tr, te = split[name]
    report = {
        'dataset': name,
        'duplication_audit': dup_audit[name],
        'strict_total': int(len(strict[name])),
        'strict_per_class': {k: int(v) for k, v in strict[name][LABEL_COLUMN].value_counts().items()},
        'train_per_class': {k: int(v) for k, v in tr[LABEL_COLUMN].value_counts().items()},
        'test_per_class': {k: int(v) for k, v in te[LABEL_COLUMN].value_counts().items()},
    }
    path = config.RESULTS_DIR / f'{name}_dedup_audit.json'
    path.write_text(json.dumps(report, indent=2))
    print('saved ->', path)

saved -> /home/koala/lab/adversec/results/ciciov2024_dedup_audit.json
saved -> /home/koala/lab/adversec/results/road_dedup_audit.json
